# Swiss open data punctuality pipeline

Same logic as `Chuuchuu_pipeline.ipynb`, but on the **Swiss open data** (SBB *Ist-Daten*, one parquet per month on the shared drive,
`Swiss open data/ist-daten-2025-MM.parquet`). One notebook run = one month; `run_swiss_pipeline.py` runs the 12 months with papermill.

What differs from the Chuuchuu pipeline, and why:

| Chuuchuu pipeline | This notebook |
|---|---|
| several agencies, UIC stop ids | one source (Switzerland), `BPUIC` stop ids, all Swiss stops |
| Step 3 deduplication (cross-agency, twin numbers) | **not needed** (one source). Only exact repeated rows are dropped |
| Step 4 operator rules (SQL translated to pandas) | the feed carries the operator (`BETREIBER_ABK`): only normalised here |
| cancellation flags often null, inferred in Step 5 | `FAELLT_AUS_TF` is always set per stop: no inference |
| journeys split per country, joint operations | every stop is Swiss: one unit per journey (columns kept so sheets stay comparable) |
| 7 per-operator sheets | **dropped**. Sheets: `Summary`, `Summary_partial_alt` and the new `Summary_all_stops` |
| punctuality at the terminus only | same, **plus** `Summary_all_stops`: punctuality at every stop, 3 min threshold, a cancelled stop counts as delayed |

Steps: 0 load - 1 stop-level preparation - 2 terminus identification - 3 operator and service flags - 4 cancellation - 5 punctuality units & summaries - 6 export - 7 sanity checks.

In [1]:
# Parameters
data_selection = "2025_01"      # month to process, e.g. "2025_01"
export_enriched = "n"           # "y" writes the per-stop enriched parquet next to the raw data (processed_data/)
export_summary = "y"            # "y" writes summary_stats/swiss_summary_<month>.xlsx
products = ["Zug"]              # PRODUKT_ID values kept. "Bus" and "Tram" exist (~90% of rows) : adding them needs far more memory

In [2]:
import os
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

from chuuchuu_paths import SWISS_OPEN_DATA_DIR, swiss_raw_path

# --- Methodology choices, all in one place -------------------------------------------------------
# A stop's delay only counts when the feed says it was *measured*. PROGNOSE / GESCHAETZT / UNBEKANNT are forecasts
# or unknowns, never confirmed by the train actually reaching the stop.
MEASURED_STATUSES = {"REAL"}
DELAY_THRESHOLDS_MIN = [3, 5, 6, 10, 15, 30, 60]     # terminus summary (same as Chuuchuu)
ALL_STOPS_THRESHOLD_MIN = 3                           # all-stops sheet
# The feed only holds Swiss stops, so a journey's other end can't be seen: journey_type is a proxy by product
INTERNATIONAL_ROUTE_TYPES = {"EC", "ICE", "TGV", "RJX", "NJ", "EN", "FLX"}
FOREIGN_OPERATOR_ABK = {"SBB GMBH", "DB", "DB REGIO", "TN"}   # cross-border operators seen in the feed (upper case)
HIGHSPEED_ROUTE_TYPES = {"ICE", "TGV", "RJX"}
NIGHT_ROUTE_TYPES = {"EN", "NJ"}


def report_memory(label):
    gb = data.memory_usage(deep=True).sum() / 1e9
    print(f"[memory] {label}: {gb:.2f} GB (data)")

# Step 0 - Raw data loading

The raw file holds every Swiss public transport stop (trains, buses, trams: ~70 M rows a month), all columns as text, empty string for missing values.
Only the kept `products` are read (filter pushed down to the parquet scan) and only the useful columns.

In [3]:
raw_path = swiss_raw_path(data_selection)
RAW_TO_NAME = {
    "BETRIEBSTAG": "date_raw", "FAHRT_BEZEICHNER": "trip_id", "BETREIBER_ABK": "operator_abk",
    "PRODUKT_ID": "product", "VERKEHRSMITTEL_TEXT": "routeType", "LINIEN_TEXT": "line",
    "ZUSATZFAHRT_TF": "extra_train", "FAELLT_AUS_TF": "cancelled_raw", "DURCHFAHRT_TF": "pass_through",
    "BPUIC": "stopId", "HALTESTELLEN_NAME": "stopName",
    "ANKUNFTSZEIT": "plannedArrival", "AN_PROGNOSE": "arrival", "AN_PROGNOSE_STATUS": "arrival_status",
    "ABFAHRTSZEIT": "plannedDeparture", "AB_PROGNOSE": "departure", "AB_PROGNOSE_STATUS": "departure_status",
}
product_variants = sorted({v for p in products for v in (p, p.upper(), p.lower(), p.capitalize())})

try:
    table = ds.dataset(raw_path).to_table(columns=list(RAW_TO_NAME), filter=ds.field("PRODUKT_ID").isin(product_variants))
except FileNotFoundError as e:
    raise FileNotFoundError(f"No Swiss open data parquet at {raw_path}") from e

data = table.to_pandas().rename(columns=RAW_TO_NAME)
del table
print(f"Loaded {raw_path.name}: {len(data):,} rows, products {data['product'].value_counts().to_dict()}")
row_counts = {"initial": len(data)}
report_memory("after load (Step 0)")

Loaded ist-daten-2025-01.parquet: 4,976,292 rows, products {'Zug': 4976292}
[memory] after load (Step 0): 1.42 GB (data)


# Step 1 - Stop-level preparation

- **Parse the times.** Local Swiss time, `dd.mm.yyyy HH:MM` (planned, minute resolution) and `dd.mm.yyyy HH:MM:SS` (forecast / actual, second resolution). Kept as local time: planned and actual are in the same clock, so the delay is right (only the one DST night per year could be off by an hour).
- **Drop pass-through rows** (`DURCHFAHRT_TF`): the train doesn't stop there, there is no stop to be punctual at.
- **Drop exact repeated rows** (same value in every column). This is *not* the Chuuchuu deduplication phase (one source here), just exact copies.
- **Delay in seconds** = actual - planned, only where the status is in `MEASURED_STATUSES`, otherwise `NaN` (unknown, never 0).
- **Country** from the UIC prefix of `BPUIC` (same table as the Chuuchuu pipeline).
- **Row order** within a train is not given by a sequence column. Planned time orders the stops; the position in the file (which follows the route, checked: 99.9% of trains) breaks the ties left by minute resolution (2% of trains).

In [4]:
before = len(data)
data = data[data["pass_through"].str.lower() != "true"]
print(f"{before - len(data):,} pass-through rows dropped")
row_counts["after_pass_through"] = len(data)

before = len(data)
data = data.drop_duplicates().reset_index(drop=True)
print(f"{before - len(data):,} exact repeated rows dropped")
row_counts["after_exact_duplicates"] = len(data)

data["pos"] = np.arange(len(data), dtype="int64")        # file order, final tie-breaker for the stop order

FMT_MIN, FMT_SEC = "%d.%m.%Y %H:%M", "%d.%m.%Y %H:%M:%S"
data["date"] = pd.to_datetime(data["date_raw"], format="%d.%m.%Y")
for col in ["plannedArrival", "plannedDeparture"]:
    data[col] = pd.to_datetime(data[col], format=FMT_MIN, errors="coerce")
for col in ["arrival", "departure"]:
    data[col] = pd.to_datetime(data[col], format=FMT_SEC, errors="coerce")
data = data.drop(columns=["date_raw"])

data["routeType"] = data["routeType"].str.strip().str.upper().replace("", "UNKNOWN")
data["operator_abk"] = data["operator_abk"].str.strip()
data["cancelled"] = data["cancelled_raw"].str.lower() == "true"
data["extra_train"] = data["extra_train"].str.lower() == "true"

# delay in seconds, measured only
for kind in ["arrival", "departure"]:
    measured = data[f"{kind}_status"].isin(MEASURED_STATUSES)
    planned = data["plannedArrival" if kind == "arrival" else "plannedDeparture"]
    data[f"{kind}Delay"] = (data[kind] - planned).dt.total_seconds().where(measured)
    print(f"{kind}Delay: {data[f'{kind}Delay'].notna().mean() * 100:.1f}% of rows measured, "
          f"{(data[kind + '_status'].isin(['PROGNOSE', 'GESCHAETZT', 'UNBEKANNT'])).mean() * 100:.1f}% forecast/unknown")

# country
uic_codes = pd.read_csv("sup_data/UIC_country_codes.csv", sep=";")
uic_codes["Numerical code"] = uic_codes["Numerical code"].astype(str)
data["country"] = data["stopId"].str[:2].map(uic_codes.drop_duplicates("Numerical code").set_index("Numerical code")["Country"])
data.loc[data["stopId"].str.len() != 7, "country"] = "ERROR_INVALID_STOP_ID"
data["country"] = data["country"].fillna("UNKNOWN_COUNTRY")
print(data["country"].value_counts(dropna=False).to_dict())

for col in ["product", "routeType", "operator_abk", "country", "line", "stopId", "arrival_status", "departure_status"]:
    data[col] = data[col].astype("category")
data = data.drop(columns=["cancelled_raw", "pass_through"])
report_memory("after Step 1")

829 pass-through rows dropped
201 exact repeated rows dropped
arrivalDelay: 86.2% of rows measured, 5.3% forecast/unknown
departureDelay: 86.2% of rows measured, 5.3% forecast/unknown
{'Switzerland': 4975262}
[memory] after Step 1: 0.63 GB (data)


# Step 2 - Terminus identification

Same rule as the Chuuchuu pipeline: a journey is `trip_id` + operating day (`journey_id`). Stops are ranked by `sort_time` (planned arrival, else planned departure); first = `depart`, last = `terminus`, the rest `intermediate`.
A journey stays `unknown` when it has a single stop or when any of its stops has no usable time. Repeated stops (loops) are legitimate here, `trip_id` is a real trip identifier and ties are resolved by file order, so the Chuuchuu "ambiguous trip" exclusion isn't needed.

In [5]:
data["sort_time"] = data["plannedArrival"].fillna(data["plannedDeparture"])
data["journey_id"] = data["date"].dt.strftime("%Y-%m-%d") + "|" + data["trip_id"].astype(str)
data = data.drop(columns=["trip_id"])
data["jcode"] = pd.factorize(data["journey_id"])[0]
data = data.sort_values(["jcode", "sort_time", "pos"], kind="stable").reset_index(drop=True)

j = data["jcode"].to_numpy()
is_first = np.r_[True, j[1:] != j[:-1]]
is_last = np.r_[j[1:] != j[:-1], True]

has_missing_time = data["sort_time"].isna().groupby(data["jcode"]).transform("any")
n_stops_in_journey = data.groupby("jcode")["pos"].transform("size")
usable = ~has_missing_time & (n_stops_in_journey > 1)

data["depart_terminus"] = np.select(
    [usable & is_first, usable & is_last, usable], ["depart", "terminus", "intermediate"], default="unknown"
)
print(data["depart_terminus"].value_counts().to_dict())
print(f"{(data['depart_terminus'] == 'unknown').mean() * 100:.2f}% of rows have unknown depart_terminus")

{'intermediate': 4067880, 'depart': 447022, 'terminus': 447022, 'unknown': 13338}
0.27% of rows have unknown depart_terminus


# Step 3 - Operator and service flags

(Chuuchuu Steps 4 and 6.) The feed gives the operator of each trip, so the SQL operator rules aren't needed: `BETREIBER_ABK` is only cleaned (`BLS-bls` -> `BLS`, `AVA-wsb` -> `AVA`, ...). All stops are Swiss, so `operator_for_punctuality` = `normalized_operator` and each journey is a single national unit.

`journey_type` is a **proxy** (the feed has no foreign stops): `international` for cross-border products (EC, ICE, TGV, RJX, NJ, EN, FLX) and cross-border operators, `domestic` otherwise. It's only a label in the summary keys, it doesn't change any count.

`HSR_service`, `night_train` and `LD` follow the Chuuchuu definitions (long distance = operator/route type in `sup_data/operators_route_types_LD.csv`, plus every high speed and night train).

In [6]:
abk = data["operator_abk"].astype(str)
data["normalized_operator"] = abk.str.split("-").str[0].str.strip()
data["normalized_operator"] = data["normalized_operator"].replace({"": np.nan})
data["operator_for_punctuality"] = data["normalized_operator"]

international = data["routeType"].isin(INTERNATIONAL_ROUTE_TYPES) | abk.str.upper().isin(FOREIGN_OPERATOR_ABK)
data["journey_type"] = np.where(international, "international", "domestic")
data["HSR_service"] = np.where(data["routeType"].isin(HIGHSPEED_ROUTE_TYPES), "Yes", "No")
data["night_train"] = np.where(data["routeType"].isin(NIGHT_ROUTE_TYPES), "Yes", "No")

ld_lookup = pd.read_csv("sup_data/operators_route_types_LD.csv")
ld_operator_key = ld_lookup["operator"].astype(str).str.strip().str.upper().replace("OBB", "OEBB")
ld_keys = set(ld_operator_key + "||" + ld_lookup["route_type_LD"].astype(str).str.strip().str.upper())
combo_key = data["normalized_operator"].astype(str).str.strip().str.upper() + "||" + data["routeType"].astype(str)
data["LD"] = np.where(combo_key.isin(ld_keys), "Yes", "No")
data.loc[(data["HSR_service"] == "Yes") | (data["night_train"] == "Yes") | (data["routeType"] == "RJ"), "LD"] = "Yes"

print(f"{data['normalized_operator'].isna().mean() * 100:.3f}% of rows without operator")
print(data.groupby("normalized_operator", observed=True).size().sort_values(ascending=False).head(15).to_dict())
print(data.groupby(["routeType", "LD"], observed=True).size().unstack(fill_value=0).sort_values("Yes", ascending=False).head(12))
report_memory("after Step 3")

0.000% of rows without operator
{'SBB': 2029017, 'BLS': 497040, 'THURBO': 429765, 'AVA': 201936, 'SOB': 188764, 'RhB': 169992, 'AB': 156266, 'RBS': 149921, 'ASM': 120680, 'TPC': 114908, 'ZB': 96324, 'SZU': 87132, 'TPF': 80689, 'MGB': 74594, 'TRN': 69004}
LD             No     Yes
routeType                
IR         103747  133841
IC              0  100238
EC              0   13198
ICE             0    5322
TGV             0    1787
RJX             0    1043
NJ              0     764
AG             15       0
EXT          1365       0
ATZ          4876       0
FLX            33       0
PE          37427       0
[memory] after Step 3: 1.69 GB (data)


# Step 4 - Cancellation status

`FAELLT_AUS_TF` is set on every row, per stop (partial cancellations exist: a train can run and skip some stops). So `arrivalCancelled_resolved` / `departureCancelled_resolved` are simply that flag in the Chuuchuu `'t'` / `'f'` coding, with no null inference.

In [7]:
data["arrivalCancelled_resolved"] = np.where(data["cancelled"], "t", "f")
data["departureCancelled_resolved"] = data["arrivalCancelled_resolved"]
print(f"{data['cancelled'].mean() * 100:.2f}% of stops cancelled")

1.17% of stops cancelled


# Step 5 - Punctuality units and summaries

## 5a - Terminus punctuality (same metrics as the Chuuchuu `Summary` / `Summary_partial_alt` sheets)

One unit per journey (all stops are on the Swiss network). Definitions are unchanged: full / partial cancellation over the stops of the unit; the closing stop is the `terminus`; delayed at X min = arrived (closing stop not cancelled) and arrival delay strictly above X minutes; a partially cancelled unit is never counted as delayed in the main measurement (`Summary`), and goes through the delay check if it reached its terminus in the alternative one (`Summary_partial_alt`).

Added column **N train arriving at terminus with measured delay**: the closing stop was reached *and* its delay was `REAL`. Chuuchuu's `N train arriving at terminus` includes arrivals whose delay is unknown, which counts them as on time; the extra column lets you compute the share on a strictly measured base.

In [8]:
clean_data = data
service_date = clean_data["date"]
year = service_date.dt.year.to_numpy()
month = service_date.dt.month.to_numpy()

grouped = clean_data.groupby("jcode")
first_rows = clean_data.drop_duplicates("jcode").set_index("jcode")
journeys = first_rows[["journey_id", "routeType", "journey_type", "HSR_service", "LD", "night_train"]].copy()
journeys["year"] = first_rows["date"].dt.year
journeys["month"] = first_rows["date"].dt.month
journeys["operator"] = first_rows["operator_for_punctuality"].astype(object)
journeys["service_operator"] = first_rows["normalized_operator"].astype(object)
journeys["punctuality_scope"] = "national network"
journeys["stat_unit_id"] = journeys["journey_id"]

journeys["n_stops"] = grouped.size()
journeys["n_cancelled_stops"] = grouped["cancelled"].sum()
journeys["is_fully_cancelled"] = journeys["n_cancelled_stops"] == journeys["n_stops"]
journeys["is_partially_cancelled"] = (journeys["n_cancelled_stops"] > 0) & (journeys["n_cancelled_stops"] < journeys["n_stops"])

# closing stop = the terminus; a journey without resolved terminus has no closing stop (NaN, as in Chuuchuu)
terminus_rows = clean_data[clean_data["depart_terminus"] == "terminus"].set_index("jcode")
assert terminus_rows.index.is_unique, "a journey has more than one terminus"
journeys["cancelled_terminus"] = terminus_rows["cancelled"].reindex(journeys.index)            # NaN where unresolved
journeys["terminus_arrival_delay"] = terminus_rows["arrivalDelay"].reindex(journeys.index)
journeys["arrived_at_terminus"] = journeys["cancelled_terminus"].eq(False)
journeys["arrived_at_terminus_measured"] = journeys["arrived_at_terminus"] & journeys["terminus_arrival_delay"].notna()

not_partially_cancelled = ~journeys["is_partially_cancelled"]
for threshold in DELAY_THRESHOLDS_MIN:
    journeys[f"delayed_{threshold}min"] = (
        journeys["arrived_at_terminus"] & not_partially_cancelled & (journeys["terminus_arrival_delay"] > threshold * 60)
    )
is_partial = journeys["is_partially_cancelled"]
journeys["is_partial_not_reaching_terminus"] = is_partial & ~journeys["arrived_at_terminus"]
journeys["is_partial_reaching_terminus"] = is_partial & journeys["arrived_at_terminus"]
for threshold in DELAY_THRESHOLDS_MIN:
    journeys[f"delayed_alt_{threshold}min"] = (
        journeys["arrived_at_terminus"] & (journeys["terminus_arrival_delay"] > threshold * 60)
    )
journeys = journeys.reset_index(drop=True)
journeys["operator"] = journeys["operator"].fillna("UNKNOWN_OPERATOR")

print(f"{len(journeys):,} punctuality units")
print(f"{journeys['cancelled_terminus'].isna().sum():,} units with no resolved closing stop")
print(f"{journeys['arrived_at_terminus'].sum():,} arrived at the terminus, {journeys['arrived_at_terminus_measured'].sum():,} with a measured delay")
print(f"{journeys['is_fully_cancelled'].sum():,} fully / {journeys['is_partially_cancelled'].sum():,} partially cancelled")
print(", ".join(f"{journeys[f'delayed_{t}min'].sum():,} delayed >{t}min" for t in DELAY_THRESHOLDS_MIN))

460,360 punctuality units
13,338 units with no resolved closing stop
438,564 arrived at the terminus, 406,019 with a measured delay
4,999 fully / 7,468 partially cancelled
20,544 delayed >3min, 7,534 delayed >5min, 5,302 delayed >6min, 2,131 delayed >10min, 1,009 delayed >15min, 263 delayed >30min, 46 delayed >60min


In [9]:
SUMMARY_KEYS = ["year", "month", "operator", "punctuality_scope", "routeType", "journey_type", "HSR_service", "LD", "night_train"]
SUMMARY_AGGREGATIONS = dict(
    n_total_train=("stat_unit_id", "size"),
    n_cancelled_full=("is_fully_cancelled", "sum"),
    n_cancelled_partial=("is_partially_cancelled", "sum"),
    **{f"n_delay_{t}min": (f"delayed_{t}min", "sum") for t in DELAY_THRESHOLDS_MIN},
    n_arriving_terminus=("arrived_at_terminus", "sum"),
    n_arriving_terminus_measured=("arrived_at_terminus_measured", "sum"),
    n_partial_reaching_terminus=("is_partial_reaching_terminus", "sum"),
    n_partial_not_reaching_terminus=("is_partial_not_reaching_terminus", "sum"),
    **{f"n_delay_alt_{t}min": (f"delayed_alt_{t}min", "sum") for t in DELAY_THRESHOLDS_MIN},
)

summary = journeys.groupby(SUMMARY_KEYS, dropna=False, observed=True).agg(**SUMMARY_AGGREGATIONS).reset_index()
summary["n_cancelled_full_or_partial"] = summary["n_cancelled_full"] + summary["n_cancelled_partial"]
summary["n_cancelled_full_or_partial_not_reaching"] = summary["n_cancelled_full"] + summary["n_partial_not_reaching_terminus"]
assert summary["n_total_train"].sum() == len(journeys), "grouped total doesn't match the unit count : a group key must be dropping nulls"

SUMMARY_COLUMN_NAMES = {
    "year": "Year", "month": "Month", "punctuality_scope": "Punctuality scope", "routeType": "Route type",
    "journey_type": "Journey type", "HSR_service": "HSR service", "LD": "Long distance", "night_train": "Night train",
    "n_total_train": "N total train",
    "n_cancelled_full": "N cancelled train (full cancellation)",
    "n_cancelled_partial": "N cancelled train (partial cancellation)",
    "n_cancelled_full_or_partial": "N cancelled train (full + partial)",
    **{f"n_delay_{t}min": f"N train delay (> {t} min)" for t in DELAY_THRESHOLDS_MIN},
    "n_arriving_terminus": "N train arriving at terminus",
    "n_arriving_terminus_measured": "N train arriving at terminus with measured delay",
    "n_partial_reaching_terminus": "N partially cancelled train reaching terminus",
    "n_partial_not_reaching_terminus": "N partially cancelled train not reaching terminus",
    "n_cancelled_full_or_partial_not_reaching": "N cancelled train (full + partial not reaching terminus)",
    **{f"n_delay_alt_{t}min": f"N train delay (> {t} min) [partial reaching terminus included]" for t in DELAY_THRESHOLDS_MIN},
}
SUMMARY_KEY_COLUMNS = ["Year", "Month", "operator", "Punctuality scope", "Route type", "Journey type", "HSR service", "Long distance", "Night train"]
SUMMARY_COUNT_COLUMNS = (
    ["N total train", "N cancelled train (full cancellation)", "N cancelled train (partial cancellation)", "N cancelled train (full + partial)"]
    + [f"N train delay (> {t} min)" for t in DELAY_THRESHOLDS_MIN]
    + ["N train arriving at terminus", "N train arriving at terminus with measured delay"]
)
SUMMARY_ALT_COUNT_COLUMNS = (
    ["N total train", "N cancelled train (full cancellation)", "N cancelled train (partial cancellation)",
     "N partially cancelled train reaching terminus", "N partially cancelled train not reaching terminus",
     "N cancelled train (full + partial not reaching terminus)"]
    + [f"N train delay (> {t} min) [partial reaching terminus included]" for t in DELAY_THRESHOLDS_MIN]
    + ["N train arriving at terminus", "N train arriving at terminus with measured delay"]
)


def format_summary(table, count_columns):
    table = table.rename(columns=SUMMARY_COLUMN_NAMES)
    return table[SUMMARY_KEY_COLUMNS + count_columns].sort_values(SUMMARY_KEY_COLUMNS)


summary_partial_alt = format_summary(summary, SUMMARY_ALT_COUNT_COLUMNS)
summary = format_summary(summary, SUMMARY_COUNT_COLUMNS)
summary

,Year,Month,operator,Punctuality scope,Route type,Journey type,HSR service,Long distance,Night train,N total train,N cancelled train (full cancellation),N cancelled train (partial cancellation),N cancelled train (full + partial),N train delay (> 3 min),N train delay (> 5 min),N train delay (> 6 min),N train delay (> 10 min),N train delay (> 15 min),N train delay (> 30 min),N train delay (> 60 min),N train arriving at terminus,N train arriving at terminus with measured delay
0,2025,1,AB,national network,EXT,domestic,No,No,No,3,1,0,1,0,0,0,0,0,0,0,2,1
1,2025,1,AB,national network,S,domestic,No,No,No,11561,2,150,152,847,218,123,26,7,0,0,11480,9742
2,2025,1,ASM,national network,S,domestic,No,No,No,9735,14,35,49,222,62,44,7,2,0,0,9704,9478
3,2025,1,AVA,national network,S,domestic,No,No,No,11823,7,29,36,307,69,36,9,4,0,0,11799,11555
4,2025,1,BLM,national network,R,domestic,No,No,No,2795,0,0,0,0,0,0,0,0,0,0,2795,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89,2025,1,ZB,national network,IR,domestic,No,No,No,1160,2,1,3,30,10,10,6,3,0,0,1157,1157
90,2025,1,ZB,national network,PE,domestic,No,No,No,1060,0,4,4,7,4,2,0,0,0,0,1059,1059
91,2025,1,ZB,national network,R,domestic,No,No,No,2737,0,0,0,26,12,11,7,6,1,0,2737,1945
92,2025,1,ZB,national network,S,domestic,No,No,No,5778,17,47,64,52,25,16,4,1,0,0,5730,5730


## 5b - Punctuality at all stops (new sheet `Summary_all_stops`)

Every stop of every train, not only the terminus:

- **Stop delay** = arrival delay (actual - planned arrival) at every stop that has a planned arrival; at a journey's first stop, which has none, the departure delay is used. Only `REAL` measurements count.
- **Delayed stop** = measured delay strictly above 3 minutes (same strict rule as the terminus metric; the feed has second resolution, so exactly 3:00 is not delayed).
- **A cancelled stop is a delayed stop**: it counts as *not on time*, whatever the delay field says.
- **Stops without a measured delay** (not cancelled, delay `PROGNOSE` / `GESCHAETZT` / `UNBEKANNT` / empty) can't be judged: they're counted separately and **excluded from the denominator**. Check `N stops not measured` to see how much that is.

`Punctuality (3 min, all stops)` = 1 - (delayed + cancelled stops) / (measured + cancelled stops).

In [10]:
has_arrival = data["plannedArrival"].notna()
stop_delay = data["arrivalDelay"].where(has_arrival, data["departureDelay"])
stop_cancelled = data["cancelled"]
stop_measured = stop_delay.notna() & ~stop_cancelled
stop_delayed = stop_measured & (stop_delay > ALL_STOPS_THRESHOLD_MIN * 60)

stops = pd.DataFrame({
    "Year": year, "Month": month,
    "operator": data["operator_for_punctuality"].astype(object).fillna("UNKNOWN_OPERATOR").to_numpy(),
    "Route type": data["routeType"].astype(object).to_numpy(),
    "Journey type": data["journey_type"].to_numpy(), "HSR service": data["HSR_service"].to_numpy(),
    "Long distance": data["LD"].to_numpy(), "Night train": data["night_train"].to_numpy(),
    "N stops": 1,
    "N cancelled stops": stop_cancelled.to_numpy(),
    "N stops with measured delay": stop_measured.to_numpy(),
    "N stops not measured": (~stop_measured & ~stop_cancelled).to_numpy(),
    f"N stops delayed (> {ALL_STOPS_THRESHOLD_MIN} min)": stop_delayed.to_numpy(),
})
ALL_STOPS_KEYS = ["Year", "Month", "operator", "Route type", "Journey type", "HSR service", "Long distance", "Night train"]
summary_all_stops = stops.groupby(ALL_STOPS_KEYS, dropna=False).sum().reset_index()
summary_all_stops["N stops not on time (delayed + cancelled)"] = (
    summary_all_stops[f"N stops delayed (> {ALL_STOPS_THRESHOLD_MIN} min)"] + summary_all_stops["N cancelled stops"]
)
judged = summary_all_stops["N stops with measured delay"] + summary_all_stops["N cancelled stops"]
summary_all_stops["Punctuality (3 min, all stops)"] = 1 - summary_all_stops["N stops not on time (delayed + cancelled)"] / judged.where(judged > 0)
summary_all_stops = summary_all_stops.sort_values(ALL_STOPS_KEYS)
del stops

tot = summary_all_stops[["N stops", "N cancelled stops", "N stops with measured delay", "N stops not measured", "N stops not on time (delayed + cancelled)"]].sum()
print(tot.to_dict())
print(f"all stops, 3 min : punctuality {(1 - tot.iloc[4] / (tot.iloc[2] + tot.iloc[1])) * 100:.2f}% ; {tot.iloc[3] / tot.iloc[0] * 100:.2f}% of stops not measured")
summary_all_stops.head()

{'N stops': 4975262, 'N cancelled stops': 58394, 'N stops with measured delay': 4692071, 'N stops not measured': 224797, 'N stops not on time (delayed + cancelled)': 321492}
all stops, 3 min : punctuality 93.23% ; 4.52% of stops not measured


,Year,Month,operator,Route type,Journey type,HSR service,Long distance,Night train,N stops,N cancelled stops,N stops with measured delay,N stops not measured,N stops delayed (> 3 min),N stops not on time (delayed + cancelled),"Punctuality (3 min, all stops)"
0,2025,1,AB,EXT,domestic,No,No,No,8,2,2,4,0,2,0.500000
1,2025,1,AB,S,domestic,No,No,No,156258,236,149840,6182,18760,18996,0.873424
2,2025,1,ASM,S,domestic,No,No,No,120680,287,119546,847,5776,6063,0.949405
3,2025,1,AVA,S,domestic,No,No,No,201936,242,198905,2789,7186,7428,0.962701
4,2025,1,BLM,R,domestic,No,No,No,8380,0,0,8380,0,0,NaN


# Step 6 - Exports

In [11]:
if export_summary.lower() == "y":
    os.makedirs("summary_stats", exist_ok=True)
    summary_path = f"summary_stats/swiss_summary_{data_selection}.xlsx"
    with pd.ExcelWriter(summary_path) as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        summary_partial_alt.to_excel(writer, sheet_name="Summary_partial_alt", index=False)
        summary_all_stops.to_excel(writer, sheet_name="Summary_all_stops", index=False)
    print(f"Saved to {summary_path} (sheets: Summary, Summary_partial_alt, Summary_all_stops)")

if export_enriched.lower() == "y":
    enriched_dir = SWISS_OPEN_DATA_DIR / "processed_data"
    enriched_dir.mkdir(exist_ok=True)
    enriched_path = enriched_dir / f"swiss_{data_selection}_enriched.parquet"
    data.drop(columns=["jcode"]).to_parquet(enriched_path)
    print(f"Saved to {enriched_path}")

Saved to summary_stats/swiss_summary_2025_01.xlsx (sheets: Summary, Summary_partial_alt, Summary_all_stops)


## Summary of the process

In [12]:
n_initial = row_counts["initial"]
for label, key_from, key_to in [("pass-through rows", "initial", "after_pass_through"),
                                ("exact repeated rows", "after_pass_through", "after_exact_duplicates")]:
    n = row_counts[key_from] - row_counts[key_to]
    print(f"Dropped - {label:<22}{n:>12,} rows ({n / n_initial * 100:6.2f}%)")
print(f"Final rows:{'':<23}{len(data):>12,} rows ({len(data) / n_initial * 100:6.2f}%)")

Dropped - pass-through rows              829 rows (  0.02%)
Dropped - exact repeated rows            201 rows (  0.00%)
Final rows:                          4,975,262 rows ( 99.98%)


# Step 7 - Sanity checks

In [13]:
sanity_results = []


def sanity(check_name, ok, message, level="fail"):
    status = "OK" if ok else level.upper()
    sanity_results.append({"check": check_name, "status": status, "message": message})
    print(f"[{status}] {check_name}: {message}")


sanity("1a rows accounted", len(data) == row_counts["after_exact_duplicates"], f"{len(data)} rows, {row_counts['after_exact_duplicates']} expected")
sanity("1b journeys accounted", data["jcode"].nunique() == len(journeys), f"{len(journeys)} units, {data['jcode'].nunique()} journeys")
sanity("1c summary total", summary["N total train"].sum() == len(journeys), f"summary sums to {summary['N total train'].sum()} trains, unit table has {len(journeys)}")
sanity("1d all-stops total", summary_all_stops["N stops"].sum() == len(data), f"{summary_all_stops['N stops'].sum()} stops in the sheet, {len(data)} rows")

n_depart = (data["depart_terminus"] == "depart").groupby(data["jcode"]).sum()
n_terminus = (data["depart_terminus"] == "terminus").groupby(data["jcode"]).sum()
sanity("2a one depart per journey", (n_depart <= 1).all(), f"{(n_depart > 1).sum()} journeys with several 'depart' stops")
sanity("2b one terminus per journey", (n_terminus <= 1).all(), f"{(n_terminus > 1).sum()} journeys with several 'terminus' stops")
resolved = data[data["depart_terminus"] != "unknown"]
bounds = resolved.groupby("jcode")["sort_time"].agg(["min", "max"])
dep_t = resolved.loc[resolved["depart_terminus"] == "depart"].set_index("jcode")["sort_time"]
ter_t = resolved.loc[resolved["depart_terminus"] == "terminus"].set_index("jcode")["sort_time"]
sanity("2c depart is first, terminus is last in time", ((dep_t != bounds["min"].reindex(dep_t.index)).sum() + (ter_t != bounds["max"].reindex(ter_t.index)).sum()) == 0,
       "depart / terminus sort_time are the min / max of their journey")

sanity("3a country known", (data["country"] == "Switzerland").all(), f"{(data['country'] != 'Switzerland').sum()} stops outside Switzerland or unresolved", level="warn")
sanity("3b operator known", data["normalized_operator"].notna().all(), f"{data['normalized_operator'].isna().sum()} stops without operator", level="warn")

both = journeys["is_fully_cancelled"] & journeys["is_partially_cancelled"]
sanity("4a full xor partial", both.sum() == 0, f"{both.sum()} units both fully and partially cancelled")
nested = (journeys["delayed_15min"] & ~journeys["delayed_5min"]) | (journeys["delayed_5min"] & ~journeys["arrived_at_terminus"])
sanity("4b delay flags nested", nested.sum() == 0, f"{nested.sum()} units break delayed_15min => delayed_5min => arrived")
sanity("4c partial split", (journeys["is_partial_reaching_terminus"].astype(int) + journeys["is_partial_not_reaching_terminus"].astype(int) == journeys["is_partially_cancelled"].astype(int)).all(),
       "reaching + not reaching = partially cancelled")
S = summary_all_stops
sanity("4d all-stops counts nested", ((S["N stops with measured delay"] + S["N cancelled stops"] + S["N stops not measured"]) == S["N stops"]).all(),
       "measured + cancelled + not measured = stops")
share_unmeasured = S["N stops not measured"].sum() / S["N stops"].sum()
sanity("5a measured share", share_unmeasured <= 0.10, f"{share_unmeasured * 100:.2f}% of stops have no measured delay (ceiling 10%)", level="warn")
unmeasured_terminus = 1 - journeys["arrived_at_terminus_measured"].sum() / max(journeys["arrived_at_terminus"].sum(), 1)
sanity("5b terminus measured share", unmeasured_terminus <= 0.10, f"{unmeasured_terminus * 100:.2f}% of arrivals at the terminus have no measured delay (ceiling 10%)", level="warn")

results = pd.DataFrame(sanity_results)
display(results)
assert not (results["status"] == "FAIL").any(), "at least one sanity check failed"

[OK] 1a rows accounted: 4975262 rows, 4975262 expected
[OK] 1b journeys accounted: 460360 units, 460360 journeys
[OK] 1c summary total: summary sums to 460360 trains, unit table has 460360
[OK] 1d all-stops total: 4975262 stops in the sheet, 4975262 rows
[OK] 2a one depart per journey: 0 journeys with several 'depart' stops
[OK] 2b one terminus per journey: 0 journeys with several 'terminus' stops
[OK] 2c depart is first, terminus is last in time: depart / terminus sort_time are the min / max of their journey
[OK] 3a country known: 0 stops outside Switzerland or unresolved
[OK] 3b operator known: 0 stops without operator
[OK] 4a full xor partial: 0 units both fully and partially cancelled
[OK] 4b delay flags nested: 0 units break delayed_15min => delayed_5min => arrived
[OK] 4c partial split: reaching + not reaching = partially cancelled
[OK] 4d all-stops counts nested: measured + cancelled + not measured = stops
[OK] 5a measured share: 4.52% of stops have no measured delay (ceiling 10

,check,status,message
0,1a rows accounted,OK,"4975262 rows, 4975262 expected"
1,1b journeys accounted,OK,"460360 units, 460360 journeys"
2,1c summary total,OK,"summary sums to 460360 trains, unit table has ..."
3,1d all-stops total,OK,"4975262 stops in the sheet, 4975262 rows"
4,2a one depart per journey,OK,0 journeys with several 'depart' stops
5,2b one terminus per journey,OK,0 journeys with several 'terminus' stops
6,"2c depart is first, terminus is last in time",OK,depart / terminus sort_time are the min / max ...
7,3a country known,OK,0 stops outside Switzerland or unresolved
8,3b operator known,OK,0 stops without operator
9,4a full xor partial,OK,0 units both fully and partially cancelled
